# Data Exploratory 2 — Canal de Suez (Global Fishing Watch)

Pipeline complet :
1. **Étape 1** — Extraction brute via l'API Global Fishing Watch (4Wings + Vessel Tracks)
2. **Étape 2** — Compression et encodage (Geohash niveau 9 + Douglas-Peucker)
3. **Étape 3** — Stockage structuré en Apache Parquet (partitionné par année/mois)
4. **Étape 4** — Requêtage DuckDB ultra-rapide sur les fichiers Parquet
5. **Étape 5** — Manipulation et formatage Polars pour les algorithmes ML
6. **HDBSCAN** — Détection des clusters de navires bloqués (même logique que `data_exploratory.ipynb`)
7. **Visualisation** — Carte Folium + analyse Matplotlib

Événement de référence : **Ever Given / crise Canal de Suez (mars 2021)**  
Zone : lat 29.8–31.3, lon 32.0–32.6  
Source : Global Fishing Watch API v3

In [ ]:
# Installe les dépendances si nécessaire
# !pip install python-geohash rdp requests pyarrow duckdb polars hdbscan folium matplotlib --quiet

In [ ]:
import os
import time
import json
import requests
import numpy as np
import polars as pl
import duckdb
import hdbscan
import folium
import pyarrow as pa
import pyarrow.parquet as pq
import matplotlib.pyplot as plt
import geohash as gh
from rdp import rdp
from pathlib import Path
from datetime import datetime

print(f"Polars  : {pl.__version__}")
print(f"DuckDB  : {duckdb.__version__}")
print(f"HDBSCAN : {hdbscan.__version__}")

In [ ]:
# -------------------------------------------------------------------
# Configuration globale
# -------------------------------------------------------------------

# Token d'accès GFW — à définir comme variable d'environnement
# export GFW_API_TOKEN="your_token_here"
GFW_API_TOKEN = os.environ.get("GFW_API_TOKEN", "")
if not GFW_API_TOKEN:
    raise EnvironmentError(
        "GFW_API_TOKEN non défini. "
        "Obtenez un token sur https://globalfishingwatch.org/our-apis/ "
        "et exportez-le : export GFW_API_TOKEN=..."
    )

BASE_URL = "https://gateway.api.globalfishingwatch.org/v3"
HEADERS = {
    "Authorization": f"Bearer {GFW_API_TOKEN}",
    "Content-Type": "application/json"
}

# Bounding box Canal de Suez (Phase 2 Manifold + Phase 3 PINNs)
SUEZ = {
    "lat_min": 29.8, "lat_max": 31.3,
    "lon_min": 32.0, "lon_max": 32.6
}

# Fenêtre de crise Ever Given
CRISIS_START = "2021-03-20"
CRISIS_END   = "2021-04-05"

# Répertoires de sortie
DATA_ROOT   = Path("../../data")
PARQUET_DIR = DATA_ROOT / "donnees_ais"
FEATURES_DIR = DATA_ROOT / "features"
FIGURES_DIR  = Path("../../outputs/figures")

for d in [PARQUET_DIR, FEATURES_DIR, FIGURES_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print("Configuration OK")
print(f"Zone Suez : lat [{SUEZ['lat_min']}, {SUEZ['lat_max']}] lon [{SUEZ['lon_min']}, {SUEZ['lon_max']}]")
print(f"Fenêtre crise : {CRISIS_START} → {CRISIS_END}")

## Étape 1 — Extraction brute (Requêtes API GFW)

Deux requêtes distinctes selon la phase :
- **Phase 2 (Manifold)** : API 4Wings — agrégation horaire de la présence de navires sur 10 ans
- **Phase 3 (PINNs)** : API Vessel Tracks — positions brutes (SOG, COG, lat, lon, timestamp) à haute fréquence pendant la crise

In [ ]:
# -------------------------------------------------------------------
# Phase 2 — API 4Wings : présence de navires agrégée par heure
# Endpoint : POST /v3/4wings/report
# Dataset  : public-global-vessel-presence:latest
# -------------------------------------------------------------------

def query_4wings_report(
    date_start: str,
    date_end: str,
    bbox: dict,
    temporal_resolution: str = "HOURLY"
) -> dict:
    """
    Interroge l'API 4Wings de Global Fishing Watch.
    Retourne la présence de navires agrégée (compte + heures) pour la bbox.
    
    temporal_resolution : 'HOURLY' | 'DAILY' | 'MONTHLY' | 'YEARLY'
    """
    geometry = {
        "type": "Polygon",
        "coordinates": [[
            [bbox["lon_min"], bbox["lat_min"]],
            [bbox["lon_max"], bbox["lat_min"]],
            [bbox["lon_max"], bbox["lat_max"]],
            [bbox["lon_min"], bbox["lat_max"]],
            [bbox["lon_min"], bbox["lat_min"]]
        ]]
    }
    payload = {
        "spatialResolution": "low",
        "temporalResolution": temporal_resolution,
        "groupBy": "FLAG",
        "datasets": ["public-global-vessel-presence:latest"],
        "dateRange": f"{date_start},{date_end}",
        "geometry": geometry
    }
    resp = requests.post(
        f"{BASE_URL}/4wings/report",
        headers=HEADERS,
        json=payload,
        timeout=120
    )
    resp.raise_for_status()
    return resp.json()


def parse_4wings_to_polars(response: dict) -> pl.DataFrame:
    """Convertit la réponse 4Wings en DataFrame Polars."""
    entries = response.get("entries", [])
    if not entries:
        return pl.DataFrame()
    records = [
        {
            "timestamp": e.get("date"),
            "flag": e.get("flag", "UNKNOWN"),
            "vessel_count": e.get("count", 0),
            "hours_present": float(e.get("hours", 0.0)),
        }
        for e in entries
    ]
    df = pl.DataFrame(records)
    df = df.with_columns(
        pl.col("timestamp").str.strptime(pl.Datetime, "%Y-%m-%dT%H:%M:%SZ", strict=False)
    )
    return df


# Requête sur l'année 2023 (adapter pour 10 ans en bouclant par année)
# Pour l'historique complet (Phase 2), itérer de 2014 à 2024
print("Requête 4Wings — présence navires Suez 2023 (hourly)...")
result_4wings = query_4wings_report("2023-01-01", "2023-12-31", SUEZ, "DAILY")

df_presence = parse_4wings_to_polars(result_4wings)
print(f"Lignes retournées : {len(df_presence)}")
print(f"Période couverte  : {df_presence['timestamp'].min()} → {df_presence['timestamp'].max()}")
print(df_presence.head(10))

In [ ]:
# -------------------------------------------------------------------
# Boucle sur 10 ans pour l'historique long (Phase 2 Manifold)
# Un appel par an pour respecter les limites de l'API
# -------------------------------------------------------------------

years = range(2014, 2025)   # 10 ans de données
all_presence = []

for year in years:
    print(f"  Collecte {year}...", end=" ")
    try:
        resp = query_4wings_report(
            f"{year}-01-01",
            f"{year}-12-31",
            SUEZ,
            temporal_resolution="DAILY"
        )
        df_year = parse_4wings_to_polars(resp)
        all_presence.append(df_year)
        print(f"{len(df_year)} entrées")
    except requests.HTTPError as e:
        print(f"ERREUR : {e}")
    time.sleep(1.0)   # Respecter le rate limit GFW

df_presence_full = pl.concat(all_presence)
print(f"\nHistorique complet : {len(df_presence_full):,} lignes — {df_presence_full['timestamp'].min()} → {df_presence_full['timestamp'].max()}")

In [ ]:
# -------------------------------------------------------------------
# Phase 3 — API Vessel Tracks : positions brutes haute fréquence
# Endpoint 1 : GET /v3/events (loitering = navires à l'arrêt)
# Endpoint 2 : GET /v3/vessels/{id}/tracks
# -------------------------------------------------------------------

def search_loitering_events(
    bbox: dict,
    date_start: str,
    date_end: str,
    limit: int = 200
) -> list:
    """
    Recherche les événements de mouillage (loitering) dans la bbox.
    Retourne la liste des événements avec les identifiants de navires.
    """
    params = {
        "datasets": "public-global-loitering-events:latest",
        "startDate": date_start,
        "endDate": date_end,
        "bbox": f"{bbox['lon_min']},{bbox['lat_min']},{bbox['lon_max']},{bbox['lat_max']}",
        "limit": limit,
        "offset": 0
    }
    resp = requests.get(
        f"{BASE_URL}/events",
        headers=HEADERS,
        params=params,
        timeout=60
    )
    resp.raise_for_status()
    return resp.json().get("entries", [])


def get_vessel_tracks(
    vessel_id: str,
    date_start: str,
    date_end: str
) -> list:
    """
    Récupère la trajectoire brute d'un navire (lon, lat, timestamp, SOG, COG).
    Retourne une liste de dicts, un par point AIS.

    Réponse GFW : GeoJSON FeatureCollection
      geometry.coordinates : [[lon, lat], ...]
      properties.timestamps : [epoch_ms, ...]
      properties.speed      : [nœuds, ...]
      properties.course     : [degrés, ...]
    """
    params = {
        "startDate": date_start,
        "endDate": date_end,
        "datasets": "public-global-vessel-tracks:latest",
        "fields": "lonlat,timestamp,speed,course"
    }
    resp = requests.get(
        f"{BASE_URL}/vessels/{vessel_id}/tracks",
        headers=HEADERS,
        params=params,
        timeout=120
    )
    resp.raise_for_status()
    data = resp.json()

    records = []
    for feature in data.get("features", []):
        coords     = feature.get("geometry", {}).get("coordinates", [])
        props      = feature.get("properties", {})
        timestamps = props.get("timestamps", [])
        speeds     = props.get("speed", [])
        courses    = props.get("course", [])

        for j, (lon, lat) in enumerate(coords):
            records.append({
                "vessel_id" : vessel_id,
                "lon"       : float(lon),
                "lat"       : float(lat),
                "timestamp" : timestamps[j] if j < len(timestamps) else None,
                "sog"       : float(speeds[j])   if j < len(speeds)     else None,
                "cog"       : float(courses[j])  if j < len(courses)    else None,
            })
    return records


# Étape 1 : trouver les navires bloqués pendant la crise Ever Given
print(f"Recherche événements de mouillage ({CRISIS_START} → {CRISIS_END})...")
loitering_events = search_loitering_events(SUEZ, CRISIS_START, CRISIS_END, limit=300)
print(f"Événements trouvés : {len(loitering_events)}")

# Extraire les IDs uniques de navires
vessel_ids = list({
    evt.get("vessel", {}).get("id")
    for evt in loitering_events
    if evt.get("vessel", {}).get("id")
})
print(f"Navires uniques   : {len(vessel_ids)}")

# Aperçu du premier événement
if loitering_events:
    print("\nExemple d'événement :")
    print(json.dumps(loitering_events[0], indent=2, default=str))

In [ ]:
# Étape 2 : récupérer les trajectoires haute fréquence (limité aux 30 premiers navires)
# Pour la production, traiter tous les vessel_ids

all_track_records = []
max_vessels = min(30, len(vessel_ids))

print(f"Extraction des tracks pour {max_vessels} navires...")
for i, vid in enumerate(vessel_ids[:max_vessels]):
    try:
        records = get_vessel_tracks(vid, CRISIS_START, CRISIS_END)
        all_track_records.extend(records)
        if (i + 1) % 5 == 0:
            print(f"  {i + 1}/{max_vessels} navires — {len(all_track_records):,} points collectés")
    except requests.HTTPError as e:
        print(f"  Navire {vid} : erreur {e.response.status_code}")
    time.sleep(0.5)   # Rate limit

print(f"\nTotal points AIS bruts collectés : {len(all_track_records):,}")

# Construire le DataFrame Polars
df_tracks_raw = pl.DataFrame(all_track_records)

# Convertir timestamp (epoch ms → Datetime)
df_tracks_raw = df_tracks_raw.with_columns(
    pl.col("timestamp").cast(pl.Int64).mul(1_000).cast(pl.Datetime("us")).alias("timestamp")
    if df_tracks_raw["timestamp"].dtype in (pl.Int64, pl.Float64)
    else pl.col("timestamp").str.strptime(pl.Datetime, "%Y-%m-%dT%H:%M:%SZ", strict=False)
)

print("\nAperçu des données brutes :")
print(df_tracks_raw.head(10))
print(f"\nSchema : {df_tracks_raw.schema}")

## Étape 2 — Compression et Encodage

Deux transformations pour alléger et standardiser les données :
- **Geohash niveau 9** : remplace lat/lon flottant par une chaîne de 9 caractères (~4.7m × 4.7m)
- **Douglas-Peucker** : élimine les points redondants des trajectoires (lignes droites du canal)

In [ ]:
# -------------------------------------------------------------------
# 2a — Encodage Geohash (précision 9 ≈ 4.7m × 4.7m)
# -------------------------------------------------------------------

def add_geohash(df: pl.DataFrame, precision: int = 9) -> pl.DataFrame:
    """
    Ajoute une colonne 'geohash' encodée à précision variable.
    - Précision 6 ≈ 1.2km × 0.6km   (clustering régional)
    - Précision 7 ≈ 153m × 153m     (mouillage)
    - Précision 9 ≈ 4.7m × 4.7m     (haute fréquence PINNs)
    """
    lats = df["lat"].to_list()
    lons = df["lon"].to_list()
    hashes = [
        gh.encode(lat, lon, precision=precision)
        if lat is not None and lon is not None
        else None
        for lat, lon in zip(lats, lons)
    ]
    return df.with_columns(pl.Series("geohash", hashes))


df_tracks_geo = add_geohash(df_tracks_raw, precision=9)

print("Geohash encodé (précision 9 ≈ 4.7m × 4.7m)")
print(df_tracks_geo.select(["vessel_id", "lat", "lon", "geohash", "sog", "cog"]).head(8))
print(f"\nNombre de cellules géographiques distinctes : {df_tracks_geo['geohash'].n_unique():,}")

In [ ]:
# -------------------------------------------------------------------
# 2b — Compression de Douglas-Peucker (RDP)
#
# Epsilon en degrés décimaux :
#   0.0001 ≈ 11m   (haute fidélité, crise/PINNs)
#   0.001  ≈ 111m  (manifold long terme)
# -------------------------------------------------------------------

def compress_trajectory_dp(
    group: pl.DataFrame,
    epsilon: float = 0.0001
) -> pl.DataFrame:
    """
    Applique l'algorithme Ramer-Douglas-Peucker sur la trajectoire d'un navire.
    Conserve uniquement les points de virage significatifs.
    Garde toujours le premier et le dernier point.
    """
    if len(group) < 3:
        return group
    points = group.select(["lon", "lat"]).to_numpy()
    mask   = rdp(points, epsilon=epsilon, return_mask=True)
    indices = np.where(mask)[0]
    return group[indices]


# Appliquer le DP par navire (triés chronologiquement)
compressed_parts = []
total_before = 0
total_after  = 0

for vid in df_tracks_geo["vessel_id"].unique().to_list():
    vessel_df = (
        df_tracks_geo
        .filter(pl.col("vessel_id") == vid)
        .filter(pl.col("timestamp").is_not_null())
        .sort("timestamp")
    )
    total_before += len(vessel_df)
    compressed    = compress_trajectory_dp(vessel_df, epsilon=0.0001)
    total_after  += len(compressed)
    compressed_parts.append(compressed)

df_tracks = pl.concat(compressed_parts)

compression_ratio = (1 - total_after / total_before) * 100 if total_before else 0
print(f"Points avant Douglas-Peucker : {total_before:>10,}")
print(f"Points après Douglas-Peucker : {total_after:>10,}")
print(f"Réduction                     : {compression_ratio:>8.1f}%")
print(f"\nDataFrame compressé : {df_tracks.shape}")
print(df_tracks.head(8))

## Étape 3 — Stockage structuré (Apache Parquet partitionné)

Structure de partitionnement :
```
data/donnees_ais/
├── suez/
│   ├── 2021/
│   │   ├── 03/
│   │   │   └── data.parquet   ← Ever Given
│   │   └── 04/
│   │       └── data.parquet
│   └── 2023/
│       └── ...
```

Format Parquet columnar : compression LZ4 par défaut, lectures ~10× plus rapides que CSV.

In [ ]:
# -------------------------------------------------------------------
# Sauvegarde en Parquet partitionné (année / mois)
# -------------------------------------------------------------------

def save_partitioned_parquet(
    df: pl.DataFrame,
    base_dir: Path,
    zone: str = "suez"
) -> None:
    """
    Sauvegarde le DataFrame en fichiers Parquet partitionnés
    base_dir/{zone}/{year}/{month:02d}/data.parquet
    """
    df_dated = df.with_columns([
        pl.col("timestamp").dt.year().alias("_year"),
        pl.col("timestamp").dt.month().alias("_month")
    ])

    partitions = (
        df_dated
        .select(["_year", "_month"])
        .unique()
        .sort(["_year", "_month"])
        .iter_rows(named=True)
    )

    for row in partitions:
        year, month = row["_year"], row["_month"]
        partition = (
            df_dated
            .filter((pl.col("_year") == year) & (pl.col("_month") == month))
            .drop(["_year", "_month"])
        )
        out_dir  = base_dir / zone / str(year) / f"{month:02d}"
        out_dir.mkdir(parents=True, exist_ok=True)
        out_path = out_dir / "data.parquet"
        partition.write_parquet(out_path, compression="lz4")
        print(f"  {out_path}  ({len(partition):,} lignes)")


print("Sauvegarde tracks Suez (crise Ever Given) en Parquet...")
save_partitioned_parquet(df_tracks, PARQUET_DIR, zone="suez")

# Idem pour les données de présence 4Wings (Phase 2 Manifold)
print("\nSauvegarde données 4Wings en Parquet...")
save_partitioned_parquet(df_presence_full, PARQUET_DIR, zone="suez_presence")

# Vérification de la structure créée
print("\nStructure Parquet générée :")
for p in sorted(PARQUET_DIR.rglob("*.parquet")):
    size_kb = p.stat().st_size / 1024
    print(f"  {p.relative_to(DATA_ROOT)}  ({size_kb:.1f} KB)")

## Étape 4 — Extraction ciblée avec DuckDB

DuckDB scanne les fichiers Parquet sans les charger intégralement en RAM.  
Les `glob patterns` (`*/*.parquet`) permettent de requêter toutes les partitions en une seule instruction SQL.

In [ ]:
con = duckdb.connect()

# Chemin glob pour toutes les partitions Suez
PARQUET_GLOB = str(PARQUET_DIR / "suez/*/*.parquet")

# Vue d'ensemble
stats = con.sql(f"""
    SELECT
        COUNT(*)                                         AS total_points,
        COUNT(DISTINCT vessel_id)                        AS nb_navires,
        MIN(timestamp)                                   AS debut,
        MAX(timestamp)                                   AS fin,
        ROUND(AVG(sog), 2)                               AS sog_moyen,
        SUM(CASE WHEN sog < 3.0 THEN 1 ELSE 0 END)      AS points_sog_bas
    FROM '{PARQUET_GLOB}'
""").pl()

print("Statistiques globales — Canal de Suez")
print(stats)

# Requête ciblée : navires bloqués pendant la crise Ever Given (SOG < 3 kt)
# Équivalent NOAA : navires dont SOG < 0.5 dans notebook 1
# Pour le Suez, on filtre à < 3 kt (chenaux encombrés à faible vitesse)
df_blocked = con.sql(f"""
    SELECT
        vessel_id,
        lat, lon,
        geohash,
        sog, cog,
        timestamp
    FROM '{PARQUET_GLOB}'
    WHERE sog  < 3.0
      AND lat  BETWEEN 29.8 AND 31.3
      AND lon  BETWEEN 32.0 AND 32.6
      AND timestamp BETWEEN '2021-03-23' AND '2021-03-30'
    ORDER BY vessel_id, timestamp
""").pl()

print(f"\nNavires bloqués (crise Ever Given) : {df_blocked['vessel_id'].n_unique()}")
print(f"Points filtrés                      : {len(df_blocked):,}")
print(df_blocked.head(10))

In [ ]:
# Agrégation quotidienne — input pour la matrice manifold (Phase 2)
# 13 features quotidiennes : vessel_count, SOG stats, rho, cluster metrics...

df_daily = con.sql(f"""
    SELECT
        DATE_TRUNC('day', timestamp)::DATE          AS date,
        COUNT(DISTINCT vessel_id)                   AS vessel_count,
        ROUND(AVG(sog),    3)                       AS sog_mean,
        ROUND(STDDEV(sog), 3)                       AS sog_std,
        ROUND(MEDIAN(sog), 3)                       AS sog_median,
        ROUND(
            SUM(CASE WHEN sog < 3.0 THEN 1.0 ELSE 0.0 END) / COUNT(*),
            4
        )                                           AS utilization_rate_rho,
        COUNT(DISTINCT geohash)                     AS distinct_cells
    FROM '{PARQUET_GLOB}'
    GROUP BY 1
    ORDER BY 1
""").pl()

print("Features quotidiennes (extrait matrice manifold) :")
print(df_daily)

## Étape 5 — Formatage pour l'IA (Polars)

Polars (construit en Rust) traite les colonnes en parallèle.  
Ici : normalisation des features, calcul de la densité portuaire ρ, structuration des tenseurs pour les PINNs/LWR.

In [ ]:
# -------------------------------------------------------------------
# 5a — Feature engineering : normalisation + densité portuaire rho
# -------------------------------------------------------------------

def zscore(col: pl.Expr) -> pl.Expr:
    return (col - col.mean()) / col.std()


df_features = df_daily.with_columns([
    # Normalisation Z-score des features continues
    zscore(pl.col("vessel_count")).alias("vessel_count_z"),
    zscore(pl.col("sog_mean")).alias("sog_mean_z"),
    zscore(pl.col("sog_std")).alias("sog_std_z"),
    zscore(pl.col("sog_median")).alias("sog_median_z"),
    zscore(pl.col("distinct_cells")).alias("distinct_cells_z"),

    # Densité portuaire rho (fraction navires bloqués = proxy de congestion)
    pl.col("utilization_rate_rho").alias("rho"),

    # Indicateur de crise : 1 si rho > 0.5 (majorité bloquée)
    (pl.col("utilization_rate_rho") > 0.5).cast(pl.Int8).alias("is_crisis"),
])

print("Features normalisées (extrait) :")
print(
    df_features.select([
        "date", "vessel_count", "vessel_count_z",
        "sog_mean", "sog_mean_z",
        "rho", "is_crisis"
    ]).head(10)
)

# Sauvegarde de la matrice de features
features_path = FEATURES_DIR / "suez_daily_features.parquet"
df_features.write_parquet(features_path, compression="lz4")
print(f"\nMatrice features sauvegardée : {features_path}")

In [ ]:
# -------------------------------------------------------------------
# 5b — Structuration des tenseurs pour les PINNs / équations LWR
#
# Format attendu par le PINN :
#   x(t) = [lat_norm, lon_norm, t_norm]  — variables spatiotemporelles
#   u(t) = [rho_norm, sog_norm]          — variables d'état (densité, vitesse)
# -------------------------------------------------------------------

df_tensor = df_blocked.with_columns([
    # Normalisation spatiale dans la bbox Suez → [0, 1]
    ((pl.col("lat") - 29.8) / (31.3 - 29.8)).alias("lat_norm"),
    ((pl.col("lon") - 32.0) / (32.6 - 32.0)).alias("lon_norm"),

    # SOG normalisée (0–20 nœuds pour le Suez)
    (pl.col("sog") / 20.0).clip(0.0, 1.0).alias("sog_norm"),

    # COG en radians (pour les équations de transport LWR)
    (pl.col("cog") * np.pi / 180.0).alias("cog_rad"),

    # Timestamp normalisé : secondes depuis le début de la crise
    (
        (pl.col("timestamp").cast(pl.Int64) -
         pl.lit(datetime(2021, 3, 23).timestamp() * 1e6).cast(pl.Int64))
        / 1_000_000.0
    ).alias("t_seconds"),
]).sort("timestamp")

print("Tenseur formaté pour PINNs/LWR :")
print(
    df_tensor.select([
        "vessel_id", "lat_norm", "lon_norm",
        "sog_norm", "cog_rad", "t_seconds"
    ]).head(10)
)
print(f"\nShape tenseur : {df_tensor.shape}")
print(f"Plage t_seconds : [{df_tensor['t_seconds'].min():.0f}s, {df_tensor['t_seconds'].max():.0f}s]")

## HDBSCAN — Clusters de navires bloqués (Canal de Suez)

Même logique que `data_exploratory.ipynb` (Houston) :
1. Une position moyenne par navire (déduplication temporelle)
2. HDBSCAN avec métrique haversine sur coordonnées en radians
3. Extraction des clusters significatifs (min 3 navires)

In [ ]:
# Déduplication : une position moyenne par navire
# (même philosophie que notebook 1 — résoudre la redondance temporelle avant clustering)
df_static = (
    df_blocked
    .group_by("vessel_id")
    .agg([
        pl.col("lat").mean().alias("lat"),
        pl.col("lon").mean().alias("lon"),
        pl.col("sog").mean().alias("sog_mean"),
        pl.col("cog").mean().alias("cog_mean"),
        pl.len().alias("nb_points"),
        pl.col("geohash").first().alias("geohash"),
    ])
)

print(f"Navires à l'arrêt (SOG < 3kt) : {len(df_static)}")
print(df_static.head(5))

# HDBSCAN sur les positions moyennes (rad pour métrique haversine)
coords     = df_static.select(["lat", "lon"]).to_numpy()
coords_rad = np.radians(coords)

clusterer = hdbscan.HDBSCAN(
    min_cluster_size=3,
    min_samples=2,
    metric="haversine",
    cluster_selection_method="eom"
)
labels       = clusterer.fit_predict(coords_rad)
probabilities = clusterer.probabilities_

n_clusters = len(set(labels)) - (1 if -1 in labels else 0)
n_noise    = (labels == -1).sum()

print(f"\nClusters HDBSCAN détectés    : {n_clusters}")
print(f"Navires bruit (non clusterisés) : {n_noise} ({100 * n_noise / len(labels):.1f}%)")
print(f"Navires en cluster              : {len(labels) - n_noise} ({100 * (len(labels) - n_noise) / len(labels):.1f}%)")

df_static = df_static.with_columns([
    pl.Series("cluster",          labels),
    pl.Series("membership_score", probabilities)
])

In [ ]:
# Résumé des clusters (top 15 par taille)
cluster_sizes = {}
for c in set(labels):
    if c == -1:
        continue
    cluster_sizes[c] = int((labels == c).sum())

print(f"Top 15 clusters — Canal de Suez (crise Ever Given mars 2021)\n")
print(f"{'Cluster':>8}  {'Navires':>8}  {'Lat centre':>12}  {'Lon centre':>12}  {'Membership':>10}")
print("-" * 60)

for c in sorted(cluster_sizes, key=cluster_sizes.get, reverse=True)[:15]:
    mask       = labels == c
    lat_c      = coords[mask, 0].mean()
    lon_c      = coords[mask, 1].mean()
    avg_member = probabilities[mask].mean()
    print(f"  C{c:>5d}  {cluster_sizes[c]:>8d}  {lat_c:>12.4f}  {lon_c:>12.4f}  {avg_member:>10.2f}")

## Visualisation — Carte Folium (Canal de Suez)

In [ ]:
# Carte Folium — même logique que notebook 1 (Houston)
# Centre : Grand Lac Amer (milieu du canal)
m = folium.Map(location=[30.5, 32.3], zoom_start=9, tiles="CartoDB positron")

# Bounding box Canal de Suez
folium.Rectangle(
    bounds=[[SUEZ["lat_min"], SUEZ["lon_min"]], [SUEZ["lat_max"], SUEZ["lon_max"]]],
    color="black", fill=False, weight=2, dash_array="5 5",
    popup="Zone Canal de Suez — AIS GFW"
).add_to(m)

colors = [
    "red", "blue", "green", "orange", "purple",
    "darkred", "cadetblue", "darkgreen", "darkblue", "pink"
]

top_clusters = sorted(cluster_sizes, key=cluster_sizes.get, reverse=True)[:15]

for i, c in enumerate(top_clusters):
    mask      = labels == c
    color     = colors[i % len(colors)]
    n_vessels = mask.sum()
    lat_c     = coords[mask, 0].mean()
    lon_c     = coords[mask, 1].mean()

    for lat, lon in zip(coords[mask, 0], coords[mask, 1]):
        folium.CircleMarker(
            [lat, lon], radius=5, color=color,
            fill=True, fill_opacity=0.7,
            popup=f"Cluster {c} — {n_vessels} navires bloqués"
        ).add_to(m)

    folium.Marker(
        [lat_c, lon_c],
        icon=folium.DivIcon(
            html=(
                f'<b style="color:{color};font-size:10px;'
                f'background:white;padding:1px;">'
                f'C{c}:{n_vessels}</b>'
            )
        )
    ).add_to(m)

# Navires bruit en gris
noise_mask = labels == -1
for lat, lon in zip(coords[noise_mask, 0], coords[noise_mask, 1]):
    folium.CircleMarker(
        [lat, lon], radius=3, color="gray",
        fill=True, fill_opacity=0.3
    ).add_to(m)

out_map = FIGURES_DIR / "suez_clusters_hdbscan.html"
m.save(str(out_map))
print(f"Carte sauvegardée : {out_map}")
m

In [ ]:
# Analyse Matplotlib : scatter spatial + distribution des tailles de clusters

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# --- Plot 1 : positions par cluster ---
ax = axes[0]
noise_mask = labels == -1

ax.scatter(
    coords[noise_mask, 1], coords[noise_mask, 0],
    c="lightgray", s=20, alpha=0.4, label="Bruit", zorder=1
)

cmap = plt.cm.get_cmap("tab20", max(n_clusters, 1))
for c in range(n_clusters):
    mask = labels == c
    if not mask.any():
        continue
    ax.scatter(
        coords[mask, 1], coords[mask, 0],
        color=cmap(c), s=50, alpha=0.85, zorder=2
    )

ax.set_xlim(SUEZ["lon_min"] - 0.05, SUEZ["lon_max"] + 0.05)
ax.set_ylim(SUEZ["lat_min"] - 0.1,  SUEZ["lat_max"] + 0.1)
ax.set_xlabel("Longitude")
ax.set_ylabel("Latitude")
ax.set_title(
    f"HDBSCAN — Navires bloqués Canal de Suez\n"
    f"{n_clusters} clusters, crise Ever Given ({CRISIS_START} → {CRISIS_END})"
)
ax.legend(loc="lower right", fontsize=8)

# --- Plot 2 : distribution des tailles de clusters ---
ax2 = axes[1]
sizes = sorted(cluster_sizes.values(), reverse=True)
ax2.bar(range(len(sizes)), sizes, color="steelblue", edgecolor="white", alpha=0.85)
ax2.axhline(
    np.median(sizes), color="orange", linestyle="--", linewidth=1.5,
    label=f"Médiane : {int(np.median(sizes))} navires"
)
ax2.set_xlabel("Rang du cluster (trié par taille décroissante)")
ax2.set_ylabel("Navires par cluster")
ax2.set_title(f"Distribution des clusters HDBSCAN ({n_clusters} clusters)")
ax2.legend()

plt.tight_layout()

out_fig = FIGURES_DIR / "suez_hdbscan_analysis.png"
plt.savefig(str(out_fig), dpi=150, bbox_inches="tight")
plt.show()
print(f"Figure sauvegardée : {out_fig}")